# <font color="steelblue">06. Gráficos para variables cualitativas y gráficos que engañan</font>

*Unidad 2. Estadística descriptiva* · Sesión teórica 6 de 30

**Autoría**:

*   Fernando Borrás Rocher (f.borras@umh.es), autor de la idea original
*   Juan Francisco Monge Ivars
*   José Santiago García Cremades

Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

Basado en [SIMUSTAT](https://unidigitalsimustat.umh.es/) (proyecto UNIDIGITAL-UMH, CC BY-SA 4.0).

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`encuesta_medios.csv`, `audiencias_tv.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["encuesta_medios.csv", "audiencias_tv.csv"]
PREGUNTAS = ["q_P06_1.json", "q_P06_2.json", "q_P06_3.json", "q_P06_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

Un buen gráfico se entiende en tres segundos; uno malo, o no se entiende o, peor, **engaña**. En las redacciones se publican a diario gráficos de barras, de sectores y de columnas. Aquí aprenderás a hacerlos con matplotlib y, sobre todo, a reconocer los trucos que distorsionan los datos.

**Qué necesitas saber antes**: el cuaderno 5 (tablas de frecuencias).

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Hacer gráficos de barras verticales y horizontales, ordenados y con etiquetas.
2. Saber cuándo un gráfico de sectores funciona y cuándo no.
3. Detectar ejes truncados y otras manipulaciones habituales.

## <font color='steelblue'>Contenidos</font>

1. El gráfico de barras
2. Comparar grupos con barras
3. Sectores: casi nunca
4. Gráficos que engañan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. El gráfico de barras</font>

In [ ]:
encuesta = pd.read_csv("encuesta_medios.csv")
audiencias = pd.read_csv("audiencias_tv.csv")

Para una variable cualitativa, el gráfico más claro es el de **barras**. Tres reglas:
1. **Ordena** las barras de mayor a menor (salvo que la variable sea ordinal).
2. Usa barras **horizontales** si las etiquetas son largas.
3. El eje de las barras **empieza en cero**.

In [ ]:
pct = (encuesta["medio_principal"].value_counts(normalize=True) * 100).sort_values()

fig, ax = plt.subplots()
ax.barh(pct.index, pct.values, color="steelblue")
for i, v in enumerate(pct.values):
    ax.text(v + 0.5, i, f"{v:.0f} %", va="center")       # etiqueta con el valor
ax.set_xlabel("% de personas")
ax.set_title("Medio principal para informarse\nEncuesta simulada, n = 1.200")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

**Qué ha pasado**: con `plt.subplots()` creamos la figura (`fig`) y los ejes (`ax`), y sobre `ax` dibujamos y ponemos títulos. Las etiquetas con el valor ahorran al lector buscar en el eje.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y marca **todas** las respuestas correctas:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P06_1.json")

## <font color='steelblue'>2. Comparar grupos con barras</font>

**Qué vamos a hacer**: comparar el share medio de cada género de programa por cadena. `groupby()` agrupa y `unstack()` pone un grupo en columnas.

In [ ]:
share = audiencias.groupby(["genero", "cadena"])["share"].mean().unstack()
share.round(1)

In [ ]:
share.plot(kind="bar", figsize=(11, 4))
plt.ylabel("Share medio (%)")
plt.title("Share medio por género y cadena (datos simulados)")
plt.xticks(rotation=0)
plt.legend(title="Cadena", bbox_to_anchor=(1, 1))
plt.show()

Con cinco cadenas y cinco géneros el gráfico ya cuesta leerlo. A veces es mejor **elegir una historia**: por ejemplo, solo los informativos.

In [ ]:
share.loc["Informativo"].sort_values().plot(kind="barh", color="steelblue")
plt.xlabel("Share medio (%)")
plt.title("Share medio de los informativos por cadena (datos simulados)")
plt.show()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P06_2.json")

## <font color='steelblue'>3. Sectores: casi nunca</font>

El gráfico de **sectores** solo funciona si hay **pocas categorías** (2 o 3) que forman **un todo**. El ojo compara mal ángulos: con cinco categorías parecidas, las barras ganan siempre.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pct.plot(kind="pie", ax=axes[0], autopct="%.0f%%")
axes[0].set_ylabel("")
axes[0].set_title("Sectores: ¿cuál es mayor, radio o papel?")
pct.plot(kind="barh", ax=axes[1], color="steelblue")
axes[1].set_title("Barras: se ve al instante")
plt.tight_layout()
plt.show()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P06_3.json")

## <font color='steelblue'>4. Gráficos que engañan</font>

El truco más común es **truncar el eje**: empezar las barras en un valor distinto de cero hace que una diferencia pequeña parezca enorme.

In [ ]:
cadenas = ["Cadena 1", "Cadena 2"]
valores = [14.2, 15.1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(cadenas, valores, color=["grey", "firebrick"])
axes[0].set_ylim(14, 15.2)
axes[0].set_title("Eje truncado: «¡arrasa!»")
axes[1].bar(cadenas, valores, color=["grey", "firebrick"])
axes[1].set_ylim(0, 16)
axes[1].set_title("Eje desde cero: casi iguales")
plt.show()

Otros trucos que debes vigilar (y no usar):

| Truco | Efecto |
| :-- | :-- |
| Eje truncado en barras | Exagera diferencias |
| Escalas distintas en dos gráficos comparados | Hace parecer iguales cosas distintas |
| Efecto 3D y perspectiva | Agranda lo que está delante |
| Iconos que crecen en área | Duplicar la altura cuadruplica el área |
| Elegir el periodo a conveniencia | Esconde la tendencia general |

En los **gráficos de líneas** sí se puede no empezar en cero, siempre que se indique claramente la escala.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P06_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Haz un gráfico de barras horizontales, ordenado, con el porcentaje de personas de cada nivel de estudios. Como es ordinal, mantén el orden Primarios → Universitarios.

<details><summary>Ver solución</summary>

```python
orden = ["Primarios", "Secundarios", "FP", "Universitarios"]
p = (encuesta["estudios"].value_counts(normalize=True) * 100).reindex(orden)
p.plot(kind="barh", color="steelblue")
plt.xlabel("%")
plt.title("Nivel de estudios (encuesta simulada, n = 1.200)")
plt.show()
```
</details>

### **Ejercicio 2** *(sección 2)*

Calcula el número medio de espectadores (en miles) por franja horaria y represéntalo en un gráfico de barras ordenado.

<details><summary>Ver solución</summary>

```python
e = audiencias.groupby("franja")["espectadores_miles"].mean().sort_values()
e.plot(kind="barh", color="steelblue")
plt.xlabel("Miles de espectadores")
plt.show()
```
</details>

### **Ejercicio 3** *(sección 3)*

Haz un gráfico de sectores con la variable `suscripcion_pago` (Sí/No). ¿Es este un caso en que el sector funciona?

<details><summary>Ver solución</summary>

```python
encuesta["suscripcion_pago"].value_counts().plot(kind="pie", autopct="%.0f%%")
plt.ylabel("")
plt.show()
```

Sí: son dos categorías que suman el 100 %.
</details>

### **Ejercicio 4** *(sección 4)*

Busca en la prensa de esta semana un gráfico con el eje truncado o con otro de los trucos de la tabla. Pega el enlace en una celda de texto y explica qué distorsiona.

<details><summary>Ver idea clave</summary>

Ejercicio abierto para comentar en clase.
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- Para variables cualitativas usa **barras**: ordenadas, horizontales si hay etiquetas largas y con el **eje desde cero**.
- `groupby()` + `mean()` resume una variable por grupos; `unstack()` lo prepara para un gráfico de barras agrupadas.
- Los **sectores** solo valen con 2–3 categorías que forman un todo.
- Desconfía de ejes truncados, escalas distintas, 3D, iconos que crecen en área y periodos elegidos a conveniencia.
- Todo gráfico debe llevar **título, unidad, fuente y número de casos**.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Eje truncado** | Eje que no empieza en cero y exagera las diferencias. |
| **`groupby`** | Agrupa las filas por los valores de una columna. |
| **Leyenda** | Recuadro que explica colores o símbolos del gráfico. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| El gráfico sale vacío o duplicado | Se mezcla `plt` con `ax` sin crear la figura | Crea `fig, ax = plt.subplots()` y dibuja sobre `ax` |
| Las etiquetas del eje se solapan | Son largas | Usa `kind="barh"` o `plt.xticks(rotation=45)` |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 7 pasarás a las variables cuantitativas: histogramas.

**Siguiente cuaderno**: [07. Gráficos para variables cuantitativas: el histograma](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P07_graficos_cuantitativas.ipynb)